# **Deep Learning for Computer Vision**

## **Project 2** - Use of Data Augmentation and Transfer Learning

CRISP-DM Phase 1 to 5

---

Authors: **António Cruz** (140129), **David Isaac** (120064), **Erik Daskalyuk** (120062), **Ivan Magalhães** (106586), **Ricardo Pereira** (120052)

Date: **2025-11-30**

&#160;

## EXTRA PARA FINE TUNING FEATURES


In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf 
from tensorflow.keras import layers, models, Input
from tensorflow.keras.applications.vgg16 import VGG16
import os


2025-11-26 11:20:30.874888: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-26 11:20:31.506746: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-26 11:20:48.778230: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Configurações Iniciais
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [3]:
# utilizar cpu para treino
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Força o uso da CPU
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  0


2025-11-26 11:20:58.722787: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [4]:

train_dir = './chest_xray/train'
test_dir = './chest_xray/test'

# Parâmetros
batch_size = 4
img_size = (256, 256)
input_shape = (256, 256, 3) # VGG16 exige 3 canais (RGB)

# Carregar treino
ds_full = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='int',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
    seed=SEED
)

# Carregar teste
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='int',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False
)

# Calcular tamanhos em batches
num_batches = tf.data.experimental.cardinality(ds_full).numpy()
val_size = int(num_batches * 0.20) # 20% para validação
train_size = num_batches - val_size

print(f"Total de batches: {num_batches}")
print(f"Utilizando {train_size} batches para TREINO e {val_size} batches para VALIDAÇÃO.")

train_dataset = ds_full.take(train_size)
val_dataset = ds_full.skip(train_size)

Found 5232 files belonging to 2 classes.
Found 159 files belonging to 1 classes.
Total de batches: 1308
Utilizando 1047 batches para TREINO e 261 batches para VALIDAÇÃO.


In [5]:
def plot_performance(history, epochs):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    
    epochs_range = range(1, epochs + 1)

    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend(loc='lower right')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.legend(loc='upper right')
    
    plt.show()


In [6]:
# Load VGG16 without the top layer (classification layers)
vgg = VGG16(include_top=False, weights='imagenet', input_shape=input_shape)

# Freeze the VGG16 layers
vgg.trainable = False

import pandas as pd
pd.set_option('max_colwidth', 100)
layers_vgg = [(layer, layer.name, layer.trainable) for layer in vgg.layers]
pd.DataFrame(layers_vgg, columns=['Layer Type', 'Layer Name', 'Layer Trainable'])

,Layer Type,Layer Name,Layer Trainable
0,"<InputLayer name=input_layer, built=True>",input_layer,False
1,"<Conv2D name=block1_conv1, built=True>",block1_conv1,False
2,"<Conv2D name=block1_conv2, built=True>",block1_conv2,False
3,"<MaxPooling2D name=block1_pool, built=True>",block1_pool,False
4,"<Conv2D name=block2_conv1, built=True>",block2_conv1,False
5,"<Conv2D name=block2_conv2, built=True>",block2_conv2,False
6,"<MaxPooling2D name=block2_pool, built=True>",block2_pool,False
7,"<Conv2D name=block3_conv1, built=True>",block3_conv1,False
8,"<Conv2D name=block3_conv2, built=True>",block3_conv2,False
9,"<Conv2D name=block3_conv3, built=True>",block3_conv3,False


In [7]:
#change epochs to 10 to reduce the training time
epochs = 2

In [8]:
# Create a new model
model = models.Sequential([
    # Use VGG16 as a feature extractor
    layers.Lambda(tf.keras.applications.vgg16.preprocess_input),
    vgg,
    
    # Flatten the output of VGG16
    layers.Flatten(),
    
    # Add new fully connected layers as proposed by the architecture presented in the previous cell
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')  # For binary classification
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Summary of the model
model.summary()

# Assuming you have `train_data` and `val_data` loaded:
# Fit the model
history = model.fit(
    train_dataset,  # Your training dataset
    validation_data=val_dataset,  # Your validation dataset
    batch_size=4,
    epochs=epochs
)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 8, 8, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/2
1047/1047 ━━━━━━━━━━━━━━━━━━━━ 461s 439ms/step - accuracy: 0.9444 - loss: 5.1845 - val_accuracy: 0.9684 - val_loss: 0.7743
Epoch 2/2
1047/1047 ━━━━━━━━━━━━━━━━━━━━ 460s 440ms/step - accuracy: 0.9754 - loss: 0.7301 - val_accuracy: 0.9674 - val_loss: 0.5396


&#160;

In [ ]:
#plot_performance(history, len(history.history['loss']), 3)